# 🏦 Banky 모델 서버 노트북 (받는 분용 안내)

이 노트북은 **챗봇의 두뇌(백엔드 + AI 모델)**를 코랩에서 띄웁니다.
실행하면 웹 화면에 넣을 **공개 주소**가 나옵니다.

## 시작 전 — 코랩 파일 탭(📁)에 아래 3개를 업로드하세요
1. `bank_form_filler.zip` — 백엔드 코드
2. 어댑터 zip (예: `Banky2_prompt_engineering.zip`) — 학습된 모델 + 규정집
3. (위 두 개면 충분합니다. 베이스 모델은 자동 다운로드됩니다.)

## 실행 순서
- **반드시 GPU 런타임**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- 위에서부터 셀을 **순서대로** 실행 (0 → 8단계)
- 마지막 **'7. 서버 실행'** 셀은 *계속 실행 중(⏳)* 상태로 둬야 합니다. 끄지 마세요.
- 거기서 나온 `🌐 주소`를 복사 → 챗봇 웹의 **[⚙️ 서버 설정]**에 붙여넣기

> 이 노트북(코랩 탭)을 닫거나 런타임이 끊기면 주소도 사라집니다.
> 체험하는 동안에는 이 탭을 켜 두세요.

---


## 0. GPU 확인

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
# T4(약 15GB)면 충분합니다. 학습 모델이 4bit라 무료 T4에도 올라갑니다.

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB


## 1. 패키지 설치
학습된 모델이 Unsloth 4bit 기반이라 peft, bitsandbytes가 필요합니다.

In [2]:
!pip install -q transformers accelerate
!pip install -q "peft==0.14.0"
!pip install -q -U "bitsandbytes>=0.46.1"
!pip install -q pymupdf langdetect
!pip install -q easyocr opencv-python-headless python-multipart
!pip install -q pyngrok nest_asyncio websockets
!pip install -q langchain langchain-community faiss-cpu sentence-transformers
print("✅ 패키지 설치 완료")
print("⚠️ 설치 후 [런타임 → 세션 다시 시작] 하고 셀 2부터 다시 실행하세요")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
ERROR: p

## 2. 백엔드 코드 업로드 (bank_form_filler.zip)
`bank_form_filler.zip`을 업로드하세요.

In [3]:
import os, sys

# 코랩 파일 탭에서 bank_form_filler.zip을 미리 업로드해두세요 (/content에 위치)
if not os.path.exists("bank_form_filler"):
    !unzip -o -q bank_form_filler.zip

sys.path.insert(0, "/content/bank_form_filler")
DATA_DIR = "/content/bank_form_filler/app/data"
!apt-get install -y -q fonts-noto-cjk > /dev/null 2>&1
print("✅ 코드 준비 완료")

✅ 코드 준비 완료


## 3. 학습된 어댑터 업로드 (Banky2_Model_Final)
`Banky2_Model_Final` 폴더를 zip으로 압축해서 업로드하거나, 드라이브를 마운트합니다.

In [4]:
import os, glob

# 어댑터(+규정집 벡터스토어)가 든 zip을 자동으로 찾아 압축 해제.
# 파일명이 무엇이든(Banky2_prompt_engineering.zip 등) 상관없이,
# 안에 adapter_config.json이 들어있는 zip을 어댑터로 인식한다.
for z in glob.glob("/content/*.zip"):
    # bank_form_filler.zip(백엔드 코드)은 건너뜀
    if "bank_form_filler" in os.path.basename(z):
        continue
    !unzip -o -q "{z}" -d /content/_adapter_extract 2>/dev/null

# adapter_config.json이 있는 폴더 = 어댑터 폴더 (폴더명 공백/한글 대응)
_cfgs = glob.glob("/content/**/adapter_config.json", recursive=True)
ADAPTER_PATH = os.path.dirname(_cfgs[0]) if _cfgs else None

# 규정집 벡터스토어 경로도 함께 탐색
_vs = glob.glob("/content/**/banky_vectorstore", recursive=True)
VECTORSTORE_PATH = _vs[0] if _vs else None

print("어댑터 경로:", ADAPTER_PATH)
print("벡터스토어 경로:", VECTORSTORE_PATH)
if ADAPTER_PATH:
    print("어댑터 파일:", os.listdir(ADAPTER_PATH))
else:
    print("❌ 어댑터를 못 찾았습니다 — 어댑터 zip을 /content에 업로드했는지 확인하세요.")
if not VECTORSTORE_PATH:
    print("⚠️ 벡터스토어 없음 — 상담(3번)은 폴백 모드로 동작합니다.")


어댑터 경로: /content/2차 인사이콘
벡터스토어 경로: /content/2차 인사이콘/banky_vectorstore
파일: ['규정집.txt', 'chat_template.jinja', 'README.md', 'tokenizer_config.json', 'banky_vectorstore', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', '뱅키모델2.ipynb']


## 4. 학습된 Qwen 로드 (Base + LoRA 어댑터)
Base(`unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit`)에 학습 어댑터를 얹습니다.
4bit라 T4에 올라갑니다. (처음엔 Base 다운로드로 몇 분)

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit"

print("1) Base 모델 로드 중... (처음엔 다운로드로 몇 분)")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float16,
)
print("2) 학습된 LoRA 어댑터 얹는 중...")
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()
print("3) 토크나이저 로드...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
print("✅ 학습된 뱅키 모델 로드 완료!")

1) Base 모델 로드 중... (처음엔 다운로드로 몇 분)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

2) 학습된 LoRA 어댑터 얹는 중...


/usr/local/lib/python3.12/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'corda_config', 'ensure_weight_tying', 'lora_ga_config', 'peft_version', 'qalora_group_size', 'target_parameters', 'trainable_token_indices', 'use_bdlora', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


3) 토크나이저 로드...
✅ 학습된 뱅키 모델 로드 완료!


> **로드 오류가 나면?**
> - `bitsandbytes` 관련 오류: 위 설치 셀을 다시 실행하고 런타임 재시작
> - 메모리 부족(OOM): 런타임 → 런타임 유형 → T4 GPU 확인, 다른 노트북 종료
> - Base 모델 다운로드가 느림: 정상입니다(7B라 처음 한 번만). 기다리세요.
> - 토크나이저 경고: 무시해도 됩니다 (어댑터 폴더의 토크나이저 사용)

## 5. 서비스 연결 (LLMEngine 주입)
로드한 뱅키를 LLMEngine에 넣으면, 추출·상담·OCR·번역이 모두 뱅키를 사용합니다.

In [6]:
import sys
sys.path.insert(0, "/content/bank_form_filler")

from app.services.llm_engine import LLMEngine
from app.services.slot_extractor_hf import HFSlotExtractor
from app.services.consult_rag import ConsultRAG, load_banky_vectorstore
import app.main as backend

engine = LLMEngine(model, tokenizer)
extractor = HFSlotExtractor(engine=engine)

# 시나리오3: 규정집 벡터스토어 로드 후 ConsultRAG에 주입
vs = load_banky_vectorstore(VECTORSTORE_PATH)   # 어댑터 업로드 셀에서 잡아둔 경로
consultant = ConsultRAG(engine=engine, vectorstore=vs)

backend.init_services(extractor=extractor, consultant=consultant)
print("✅ 서비스 연결 완료 (학습된 뱅키 사용)")
print("   - 시나리오1·2(추출):", extractor is not None)
print("   - 시나리오3(상담 RAG):", consultant is not None, "| 벡터스토어:", vs is not None)

/content/bank_form_filler/app/services/consult_rag.py:145: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 서비스 연결 완료 (학습된 뱅키 사용)
   - 시나리오1·2(추출): True
   - 시나리오3(상담 RAG): True | 벡터스토어: True


## 6. 서버 실행 + 공개 주소 생성
cloudflared로 바꿨따!!!!!!!!!! ngrok은 우우 쓰레기다

In [8]:
import os
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
print("✅ cloudflared 준비 완료")

✅ cloudflared 준비 완료


In [9]:
import nest_asyncio, asyncio, uvicorn, subprocess, re, threading, time
nest_asyncio.apply()
PORT = 8000

# 1) cloudflared 터널을 백그라운드로 시작
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

# 2) 로그에서 공개 주소 추출 (별도 스레드)
public_url = None
def _grab_url():
    global public_url
    for line in tunnel.stdout:
        m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group()
            break
threading.Thread(target=_grab_url, daemon=True).start()

# 주소 나올 때까지 대기 (최대 30초)
for _ in range(30):
    if public_url: break
    time.sleep(1)

print("="*55)
print(f"🌐 프론트에게 줄 주소: {public_url}")
print("="*55)
print("\n👉 챗봇 웹 화면의 [⚙️ 서버 설정] 버튼을 눌러 이 주소를 붙여넣으세요.")
print("\n⚠️ 이 셀은 '계속 실행 중'(로딩) 상태여야 정상입니다. 끄지 마세요.\n")

# 3) 서버 실행 (계속 실행 상태 유지)
config = uvicorn.Config(backend.app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)
await server.serve()

🌐 프론트에게 줄 주소: https://configured-badly-pacific-hometown.trycloudflare.com

👉 chatting.html의 CHATBOT_URL에 이 주소를 넣으세요 (끝에 / 없이)

⚠️ 이 셀은 '계속 실행 중'(로딩) 상태여야 정상입니다. 끄지 마세요.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/doc